# Line search & trust regions — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Globalization: make each local model step safe
Newton or gradient directions are local advice, so optimization needs a safety policy. These five examples compare Armijo backtracking, exact line search on a quadratic, a trust-region clipped step, radius expansion after a good model prediction, and radius contraction after a bad prediction.

In [ ]:
# 1) Armijo backtracking accepts the first sufficiently decreasing step
f=lambda x:(x-2)**2; g=lambda x:2*(x-2); x0=0.; p=-g(x0); alpha=1.; c=0.5
while f(x0+alpha*p)>f(x0)+c*alpha*g(x0)*p: alpha*=0.5
xx=np.linspace(-1,4,200); plt.figure(figsize=(5,3)); plt.plot(xx,f(xx)); plt.scatter([x0,x0+alpha*p],[f(x0),f(x0+alpha*p)],c=['r','g']); plt.title(f'accepted alpha={alpha}')
assert abs(alpha-0.5)<1e-12 and f(x0+alpha*p)==0

In [ ]:
# 2) exact line search on a quadratic
H=np.diag([2.,8.]); x=np.array([2.,1.]); grad=H@x; p=-grad; alpha=(grad@grad)/(grad@H@grad); xn=x+alpha*p
plt.figure(figsize=(5,4)); plt.arrow(x[0],x[1],xn[0]-x[0],xn[1]-x[1],head_width=.08); plt.scatter([0,x[0],xn[0]],[0,x[1],xn[1]]); plt.title(f'exact alpha={alpha:.3f}')
assert abs(alpha-0.14705882352941177)<1e-12 and (xn@H@xn)<(x@H@x)

In [ ]:
# 3) trust region clips an overlarge Newton step
B=np.eye(2); grad=np.array([3.,4.]); newton=-grad; Delta=2.; step=newton*min(1,Delta/np.linalg.norm(newton))
plt.figure(figsize=(4,4)); circle=plt.Circle((0,0),Delta,fill=False); plt.gca().add_patch(circle); plt.arrow(0,0,step[0],step[1],head_width=.08); plt.axis('equal'); plt.title('step clipped to radius')
assert abs(np.linalg.norm(step)-2)<1e-12

In [ ]:
# 4) good agreement expands the radius
pred=np.array([1.0,0.8,0.6]); actual=np.array([0.98,0.79,0.61]); rho=actual[-1]/pred[-1]; Delta=1.; newDelta=2*Delta if rho>0.75 else Delta
plt.figure(figsize=(5,3)); plt.plot(pred,label='predicted'); plt.plot(actual,'--',label='actual'); plt.legend(); plt.title(f'rho={rho:.3f}: expand')
assert rho>0.75 and newDelta==2

In [ ]:
# 5) bad agreement contracts the radius
pred_red=1.0; actual_red=0.1; rho=actual_red/pred_red; Delta=1.; newDelta=0.25*Delta if rho<0.25 else Delta
plt.figure(figsize=(5,3)); plt.bar(['predicted','actual'],[pred_red,actual_red]); plt.title(f'rho={rho:.2f}: contract')
assert rho<0.25 and abs(newDelta-0.25)<1e-12